# JEPA on recurrence plots: synthetic benchmark

This notebook compares simple, feature-based, random-init, and supervised baselines on six synthetic tasks. Smoke mode uses small data and a short encoder list; the full run uses the registered benchmark sizes.

## 1. Setup

In [ ]:
from pathlib import Path
import os, sys, time, json, hashlib, warnings
sys.path.insert(0, str(Path('../../src').resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, r2_score
from scipy.stats import spearmanr
from rpjepa.data import TASKS, SYSTEM_CLASSES, TaskData, check_data
from rpjepa.representations import rp_input
from rpjepa.features import rqa_features
from rpjepa.encoders import ENCODERS, make_encoder, n_params, count_flops
from rpjepa.train import train_supervised, random_init_encoder
from rpjepa.probe import embed, linear_probe
from rpjepa.experiment import SMOKE, get_split, cached_run, summarise

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N, M, TAU, D_MODEL, DEPTH = 64, 3, 1, 64, 2
SIZES = {'train': 20000, 'val': 2000, 'test': 5000, 'probe_train': 5000, 'probe_test': 5000}
SEEDS = (0, 1)
EPOCHS = 30
ENCODERS_RUN = [name for name in ENCODERS if not name.startswith('torp_')]
if SMOKE:
    SIZES = {'train': 1000, 'val': 300, 'test': 500, 'probe_train': 500, 'probe_test': 500}
    SEEDS, EPOCHS = (0,), 3
    ENCODERS_RUN = ['raw_attn', 'raw_conv', 'rp_cnn']
CACHE_ROOT = Path('../../data/processed/jepa') / ('smoke' if SMOKE else 'step1')
TABLE_ROOT = Path('../../results/tables/jepa/step1') / ('smoke' if SMOKE else '')
CACHE_ROOT.mkdir(parents=True, exist_ok=True); TABLE_ROOT.mkdir(parents=True, exist_ok=True)
print({'device': DEVICE, 'smoke': SMOKE, 'sizes': SIZES, 'seeds': SEEDS, 'encoders': ENCODERS_RUN})

## Metrics

Classification task `system` is scored by test accuracy. Regression tasks are scored by test-set R²; Spearman correlation is also stored for regression results.

## 2. Tasks and data checks

The six tasks cover system identification and regression targets. The method column indicates the intended information source, not an assumption that other methods cannot work.

In [ ]:
task_table = pd.DataFrame([
    ('system','classification','system family','nonlinear dynamics; RPs may help'),
    ('lyapunov','regression','largest Lyapunov exponent','local divergence / dynamics'),
    ('noise','regression','log noise-to-signal ratio','noise-sensitive RQA / statistics'),
    ('changepoint','regression','switch location / window length','local temporal change'),
    ('recall','regression','value associated with queried key','associative matching'),
    ('forecast','regression','future sample at horizon h=3','dynamical state and evolution'),
], columns=['task','kind','target','what_can_solve_it'])
display(task_table)
checks = check_data(); display(checks)

# One sample per system family: raw window above its distance-matrix recurrence input.
viz = get_split('system', 'probe_test', max(500, len(SYSTEM_CLASSES)), N, M, TAU, base_seed=0)
fig, axes = plt.subplots(len(SYSTEM_CLASSES), 2, figsize=(8, 2.1*len(SYSTEM_CLASSES)))
chosen = [int(np.flatnonzero(viz.y == label)[0]) for label in range(len(SYSTEM_CLASSES))]
for row, (label, idx) in enumerate(zip(SYSTEM_CLASSES, chosen)):
    axes[row,0].plot(viz.x[idx], lw=.8); axes[row,0].set_ylabel(label, rotation=0, labelpad=48)
    image = rp_input(torch.as_tensor(viz.x[idx:idx+1]), M, TAU, kind='D')[0,0].cpu().numpy()
    axes[row,1].imshow(image, cmap='viridis', origin='lower', aspect='auto')
    if row == 0: axes[row,0].set_title('Observed window'); axes[row,1].set_title('Distance-matrix RP (D)')
    if row < len(SYSTEM_CLASSES)-1: axes[row,0].set_xticklabels([]); axes[row,1].set_xticklabels([])
plt.tight_layout(); plt.show()

## 3. Baselines without learning on raw data

Trivial references predict the probe-training majority class or mean. RQA features are extracted on the probe splits and evaluated with a linear probe and histogram gradient boosting. Recall and forecast also receive task-specific analogue oracles.

In [ ]:
def data(task, split, seed, **difficulty):
    return get_split(task, split, SIZES[split], N, M, TAU, base_seed=seed, **difficulty)

def regression_scores(y, pred):
    return {'metric': float(r2_score(y, pred)), 'spearman': float(spearmanr(y, pred).statistic)}

def cached_features(task, split, seed, td, identity_extra=None):
    ident = {'task':task,'split':split,'seed':int(seed),'n':len(td.x),'n_states':N,'m':M,'tau':TAU,'difficulty':identity_extra or {}}
    digest = hashlib.sha256(json.dumps(ident,sort_keys=True).encode()).hexdigest()[:16]
    folder = CACHE_ROOT / 'features'; folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{task}_{split}_{digest}.csv'
    if path.exists(): return pd.read_csv(path)
    frame = rqa_features(td.x, m=M, tau=TAU); frame.to_csv(path,index=False); return frame

def rqa_model_result(task, seed, model_name):
    tr, te = data(task,'probe_train',seed), data(task,'probe_test',seed)
    Ftr, Fte = cached_features(task,'probe_train',seed,tr), cached_features(task,'probe_test',seed,te)
    imputer = SimpleImputer(strategy='median')
    Xtr, Xte = imputer.fit_transform(Ftr), imputer.transform(Fte)
    if model_name == 'rqa+linear':
        result = linear_probe(Xtr, tr.y, Xte, te.y, tr.kind)
    else:
        model = (HistGradientBoostingClassifier(max_iter=100, random_state=seed) if tr.kind == 'cls' else
                 HistGradientBoostingRegressor(max_iter=100, random_state=seed))
        model.fit(Xtr, tr.y); pred = model.predict(Xte)
        result = {'metric': float(accuracy_score(te.y,pred))} if tr.kind == 'cls' else regression_scores(te.y,pred)
    return result

def oracle_recall(x, y, key_len=M):
    query=x[:,-key_len:]; predictions=np.empty(len(x),dtype=np.float64)
    for i,row in enumerate(x):
        segments=np.lib.stride_tricks.sliding_window_view(row[:-key_len],key_len)
        # Exclude candidate keys that overlap the final query; each candidate's next sample is its value.
        valid=max(0,len(row)-2*key_len)
        segments=segments[:valid]; successors=row[key_len:key_len+valid]
        best=np.square(segments-query[i]).mean(axis=1).argmin(); predictions[i]=successors[best]
    # Calibrate raw predicted values with probe-training targets, not test labels.
    return predictions

def oracle_forecast(x, h=3, m=M):
    predictions=[]
    for row in x:
        states=np.lib.stride_tricks.sliding_window_view(row,m)
        last=states[-1]
        eligible=np.arange(max(0,len(states)-h))
        best=eligible[np.square(states[eligible]-last).mean(axis=1).argmin()]
        predictions.append(row[min(best+h,len(row)-1)])
    return np.asarray(predictions)

baseline_rows=[]
for task, kind in TASKS.items():
  for seed in SEEDS:
    tr, te = data(task,'probe_train',seed), data(task,'probe_test',seed)
    if kind == 'cls':
      values, counts=np.unique(tr.y,return_counts=True); pred=np.full(len(te.y),values[counts.argmax()])
      result={'metric':float(accuracy_score(te.y,pred))}
    else:
      pred=np.full(len(te.y),float(np.mean(tr.y))); result=regression_scores(te.y,pred)
    baseline_rows.append({'method':'trivial','task':task,'seed':seed,**result})
    for method in ('rqa+linear','rqa+hgb'):
      result=cached_run(CACHE_ROOT/'baselines',{'task':task,'seed':seed,'method':method,'sizes':SIZES},
                        lambda task=task,seed=seed,method=method:rqa_model_result(task,seed,method))
      baseline_rows.append({'method':method,'task':task,'seed':seed,**result})
    if task in ('recall','forecast'):
      prediction=oracle_recall(te.x,te.y) if task=='recall' else oracle_forecast(te.x)
      # Fit the oracle's affine calibration only on training examples.
      train_prediction=oracle_recall(tr.x,tr.y) if task=='recall' else oracle_forecast(tr.x)
      coef=np.polyfit(train_prediction,tr.y,1); prediction=np.polyval(coef,prediction)
      baseline_rows.append({'method':'oracle','task':task,'seed':seed,**regression_scores(te.y,prediction)})
baseline_df=pd.DataFrame(baseline_rows); display(baseline_df)

## 4. Random-initialized frozen encoders

For each selected encoder and seed, fit a linear probe to pooled embeddings from an untrained frozen network. This baseline tests whether JEPA pretraining adds value beyond architecture and random features.

In [ ]:
random_rows=[]; parameter_rows=[]
for enc in ENCODERS_RUN:
  for task,kind in TASKS.items():
    for seed in SEEDS:
      tr,te=data(task,'probe_train',seed),data(task,'probe_test',seed)
      def run_random(enc=enc,task=task,seed=seed,tr=tr,te=te,kind=kind):
        model=random_init_encoder(enc,D_MODEL,DEPTH,n_states=N,m=M,tau=TAU,seed=seed,device=DEVICE)
        ztr=embed(model,tr.x,batch=128 if SMOKE else 512,device=DEVICE); zte=embed(model,te.x,batch=128 if SMOKE else 512,device=DEVICE)
        return linear_probe(ztr,tr.y,zte,te.y,kind)
      result=cached_run(CACHE_ROOT/'random_init',{'encoder':enc,'task':task,'seed':seed,'n_states':N,'d':D_MODEL,'depth':DEPTH,'sizes':SIZES},run_random)
      random_rows.append({'method':f'random-init {enc}','encoder':enc,'task':task,'seed':seed,**result})
  probe_model=random_init_encoder(enc,D_MODEL,DEPTH,n_states=N,m=M,tau=TAU,seed=SEEDS[0],device='cpu')
  parameter_rows.append({'encoder':enc,'params':n_params(probe_model),'flops':count_flops(probe_model,N+(M-1)*TAU,batch=1)})
random_df=pd.DataFrame(random_rows); params_df=pd.DataFrame(parameter_rows)
display(random_df); display(params_df)

## 5. Supervised reference

Train each encoder with a pooled linear head using train/validation splits, early stopping, and the test split. Cached runs resume cleanly after interruption.

In [ ]:
supervised_rows=[]
for enc in ENCODERS_RUN:
  for task in TASKS:
    for seed in SEEDS:
      tr,va,te=(data(task,s,seed) for s in ('train','val','test'))
      key={'encoder':enc,'task':task,'seed':seed,'sizes':SIZES,'n_states':N,'m':M,'tau':TAU,'d':D_MODEL,'depth':DEPTH,'epochs':EPOCHS}
      result=cached_run(CACHE_ROOT/'supervised',key,lambda enc=enc,tr=tr,va=va,te=te,seed=seed:train_supervised(
          enc,D_MODEL,DEPTH,tr,va,te,n_states=N,m=M,tau=TAU,epochs=EPOCHS,batch=128,seed=seed,device=DEVICE))
      supervised_rows.append({'method':f'supervised {enc}','encoder':enc,'task':task,'seed':seed,**result})
supervised_df=pd.DataFrame(supervised_rows); display(supervised_df[['encoder','task','seed','metric','val_metric','params','flops','train_s','epochs_run']])

## 6. Recall difficulty curve

Compare nearest-key matching with RQA-feature linear probes as the number of planted pairs and query noise increase. Combinations that cannot fit are recorded as skipped.

In [ ]:
difficulty_rows=[]; skipped=[]
curve_n=300 if SMOKE else 1000
for pairs in (1,2,4,8):
  for noise in (0.0,0.1,0.3):
    for seed in SEEDS:
      try:
        tr=get_split('recall','probe_train',curve_n,N,M,TAU,base_seed=seed,n_pairs=pairs,key_len=M,query_noise=noise)
        te=get_split('recall','probe_test',curve_n,N,M,TAU,base_seed=seed,n_pairs=pairs,key_len=M,query_noise=noise)
      except ValueError as exc:
        skipped.append({'n_pairs':pairs,'query_noise':noise,'reason':str(exc)}); continue
      match=oracle_recall(te.x,te.y); match_tr=oracle_recall(tr.x,tr.y)
      slope=np.polyfit(match_tr,tr.y,1); match=np.polyval(slope,match)
      difficulty_rows.append({'method':'nearest-key oracle','n_pairs':pairs,'query_noise':noise,'seed':seed,**regression_scores(te.y,match)})
      # Use the shared feature cache protocol for each difficulty tuple.
      difficulty={'n_pairs':pairs,'key_len':M,'query_noise':noise}
      a=cached_features('recall','probe_train',seed,tr,difficulty)
      b=cached_features('recall','probe_test',seed,te,difficulty)
      imp=SimpleImputer(strategy='median'); a=imp.fit_transform(a); b=imp.transform(b)
      lin=linear_probe(a,tr.y,b,te.y,'reg')
      difficulty_rows.append({'method':'rqa+linear','n_pairs':pairs,'query_noise':noise,'seed':seed,**lin})
difficulty_df=pd.DataFrame(difficulty_rows); display(difficulty_df); display(pd.DataFrame(skipped))

## 7. System-task confusion matrix

Select the supervised encoder with the highest mean system accuracy, refit its classifier on each training seed, and inspect test confusions. The highlighted pairs distinguish Lorenz from its Fourier surrogate and logistic from its Fourier surrogate.

In [ ]:
system_scores=supervised_df[supervised_df.task=='system'].groupby('encoder').metric.mean()
best_system_encoder=system_scores.idxmax(); print('Best supervised system encoder:',best_system_encoder)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
confusions=[]
for seed in SEEDS:
  tr,te=data('system','train',seed),data('system','test',seed)
  torch.manual_seed(seed)
  model=make_encoder(best_system_encoder,D_MODEL,DEPTH,N,m=M,tau=TAU).to(DEVICE)
  head=torch.nn.Linear(2*D_MODEL,tr.n_classes).to(DEVICE); opt=torch.optim.AdamW(list(model.parameters())+list(head.parameters()),lr=1e-3)
  model.train(); head.train()
  for epoch in range(EPOCHS):
    order=np.random.default_rng(seed+epoch).permutation(len(tr.x))
    for start in range(0,len(order),128):
      idx=order[start:start+128]; xb=torch.as_tensor(tr.x[idx],device=DEVICE); yb=torch.as_tensor(tr.y[idx],device=DEVICE,dtype=torch.long)
      opt.zero_grad(set_to_none=True); tok=model(xb); logits=head(torch.cat((tok.mean(1),tok[:,-1]),dim=-1)); loss=torch.nn.functional.cross_entropy(logits,yb); loss.backward(); opt.step()
  model.eval(); pred=[]
  with torch.no_grad():
    for start in range(0,len(te.x),128):
      xb=torch.as_tensor(te.x[start:start+128],device=DEVICE); tok=model(xb); logits=head(torch.cat((tok.mean(1),tok[:,-1]),dim=-1)); pred.extend(logits.argmax(1).cpu().numpy())
  confusions.append(confusion_matrix(te.y,np.asarray(pred),labels=np.arange(len(SYSTEM_CLASSES)),normalize='true'))
cm=np.mean(confusions,axis=0); fig,ax=plt.subplots(figsize=(9,8)); disp=ConfusionMatrixDisplay(cm,display_labels=SYSTEM_CLASSES); disp.plot(ax=ax,xticks_rotation=70,cmap='Blues',values_format='.2f',colorbar=False)
for a,b in (('lorenz','ft_lorenz'),('logistic','ft_logistic')):
  i,j=SYSTEM_CLASSES.index(a),SYSTEM_CLASSES.index(b); ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,fill=False,edgecolor='red',linewidth=2)); ax.add_patch(plt.Rectangle((i-.5,j-.5),1,1,fill=False,edgecolor='red',linewidth=2))
plt.title(f'Test confusion: {best_system_encoder}'); plt.tight_layout(); plt.show()

## 8. Results

The tidy table reports mean test metric and seed standard deviation for every method/task. Encoder parameter and forward-FLOP counts are listed separately. Tables are written to the Step 1 directory; smoke outputs stay isolated.

In [ ]:
baseline_df.to_csv(TABLE_ROOT/'step1_baselines.csv',index=False)
random_df.to_csv(TABLE_ROOT/'step1_random_init.csv',index=False)
supervised_df.to_csv(TABLE_ROOT/'step1_supervised.csv',index=False)
all_rows=pd.concat([baseline_df[['method','task','seed','metric']],random_df[['method','task','seed','metric']],supervised_df[['method','task','seed','metric']]],ignore_index=True)
summary=all_rows.groupby(['method','task']).metric.agg(mean='mean',sd='std').reset_index()
summary['value']=summary.apply(lambda r:f"{r['mean']:.3f} ({r['sd']:.3f})" if pd.notna(r['sd']) else f"{r['mean']:.3f} (—)",axis=1)
summary_table=summary.pivot(index='method',columns='task',values='value').reindex(columns=list(TASKS))
display(summary_table); display(params_df)
summary.to_csv(TABLE_ROOT/'step1_summary.csv',index=False)
params_df.to_csv(TABLE_ROOT/'step1_encoder_costs.csv',index=False)
print('CSV directory:',TABLE_ROOT.resolve())

## 9. Findings

Full run: N = 64 states (m = 3, τ = 1), d = 64, depth 2, seeds 0 and 1, train/val/test 20,000/2,000/5,000, probes 5,000/5,000. Values are test accuracy (`system`) or test R² (others), mean of the two seeds.

**The benchmark behaves as designed.**
- Trivial predictors score ≈ 0 R² (`system`: 0.091 = 1/11 classes).
- `recall` is solved only by matching: the nearest-key oracle reaches 0.999, while RQA features score ≈ 0 (linear −0.003, gradient boosting −0.028). The oracle drops to ≈ 0.4 when the query noise is 0.3.
- Phase-randomised surrogates are separable from the originals, so nonlinear structure is learnable: in the best model's confusion matrix only 1–2 % of Lorenz windows are confused with their surrogates, and 2–4 % for the logistic map.
- Two ceilings come from the task design, not from the models. `system` tops out at ≈ 0.88 because AR(1), white noise and regime-switching AR overlap (an AR(1) with small φ looks like noise). `changepoint` tops out at R² ≈ 0.22 because many switches are between similar φ.

**Supervised reference (ceiling for later probes).**

| | system | lyapunov | noise | changepoint | recall | forecast | FLOPs/window |
|---|---|---|---|---|---|---|---|
| raw_attn | 0.866 | 0.943 | 0.981 | 0.205 | 0.852 | 0.978 | 8.4 M |
| raw_conv | 0.860 | 0.933 | 0.982 | −0.001 | 0.007 | 0.903 | 9.5 M |
| raw_hyena | 0.843 | 0.928 | 0.980 | 0.175 | 0.011 | 0.927 | 11.6 M |
| rp_cnn (D + series) | **0.877** | 0.925 | **0.984** | 0.112 | **0.999** | **0.983** | 1,145 M |
| rp_s4nd | 0.870 | 0.915 | 0.982 | **0.221** | 0.081 | 0.893 | 745 M |
| rp_vit | 0.834 | 0.921 | 0.979 | −0.002 | 0.011 | 0.917 | 12.1 M |
| recattn (exact) | 0.865 | **0.945** | 0.982 | 0.185 | 0.844 | 0.975 | 16.6 M |
| recattn_taylor | 0.863 | 0.945 | 0.983 | 0.174 | 0.051 | 0.972 | 18.1 M |

1. **A first look at H2 (tested properly in Step 2):** a local 3×3 CNN on the distance matrix solves recall (0.999), where the same kind of local convolution on the raw series fails (0.007) and attention gets 0.852. The price is compute: rp_cnn uses about 136× the FLOPs of raw_attn per window and took about 14× longer to train.
2. **An RP input alone is not enough.** rp_s4nd (0.081) and rp_vit (0.011) fail recall. Recall needs a cell-level match in the query row. 8×8 patches blur it, and the smooth global S4ND kernels did not learn it within the budget.
3. **The Taylor approximation breaks recurrence attention for recall.** Exact Gaussian recurrence attention matches raw attention (0.844 vs 0.852), but the Based-style 2nd-order Taylor version drops to 0.051 on recall while matching it on every other task. At initialisation the Taylor kernel differs from the exact kernel by 12–22 % on average, enough to flatten the sharp recurrence peak.
4. **Untrained encoders are strong baselines.** A frozen random-init encoder plus a linear probe reaches 0.60–0.84 on `system` and 0.58–0.85 on `lyapunov`. Several of them beat RQA features + linear (0.712, 0.796) and RQA + gradient boosting on `system` (0.739). Step 3's JEPA embeddings must beat these random-init numbers, not only the RQA features.
5. **RQA features with gradient boosting are strong on the smooth regression targets** (lyapunov 0.909, noise 0.957, forecast 0.863), but useless for recall and changepoint.

**Caveats.**
- Only two seeds.
- Early stopping with patience 5 ended some runs before they started learning: rp_cnn changepoint seed 1 stayed at 0, and rp_s4nd recall seed 0 stopped at epoch 8. That is why the rp_cnn changepoint mean (0.112) has a large SD. Step 2 uses patience 10 and 40 epochs for recall.
- The GPU was shared with other jobs during part of this run, so `train_s` (especially for rp_s4nd) is inflated. Compare costs using FLOPs, which are counted rather than timed.
